# 01 — Data audit
What does the merged match table look like, and what did cleaning change?
Run `python -m pitchiq.pipeline all` first.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))   # run from notebooks/
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from pitchiq import config
pd.set_option("display.width", 160)
print("Reading from:", config.ROOT, "| synthetic:", config.SYNTHETIC)

In [ ]:
m = pd.read_parquet(config.PROCESSED / "matches.parquet")
m.groupby("season").agg(matches=("match_id","size"), xg_coverage=("home_xg", lambda s: s.notna().mean()),
                        close_odds=("odds_close_src", lambda s: s.notna().mean())).T

Every complete season should have 380 matches. xG coverage should be ~1.0 from 2014 on and 0 before.

In [ ]:
print((config.REPORTS / "cleaning_log.md").read_text()[:3000])

In [ ]:
res = m[m.season >= 2014].groupby("season").result.value_counts(normalize=True).unstack()
res.plot(kind="bar", stacked=True, figsize=(10,4), title="H/D/A share by season"); plt.show()
res

Look at 2020/21: matches behind closed doors. Does the home-win share drop? That is a natural experiment on home advantage.

In [ ]:
odds = m.dropna(subset=["odds_close_H"])
over = (1/odds[["odds_close_H","odds_close_D","odds_close_A"]]).sum(axis=1) - 1
over.groupby(odds.season).mean().plot(title="Average closing overround (bookmaker margin)"); plt.show()